In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ----------------------------------------------------
# 1. Setup & Data Loading
# ----------------------------------------------------
# IMPORTANT: Provide your correct path below
data_dir = r"/content/"

train_ridership = pd.read_csv(f"{data_dir}Train_Ridership_2022_to_2025H1.csv")
shock_ridership = pd.read_csv(f"{data_dir}Shock_Ridership_2025_Q3.csv")
oot_ridership   = pd.read_csv(f"{data_dir}OutOfTime_Ridership_2025_Q4.csv")
bus_routes      = pd.read_csv(f"{data_dir}Bus_Routes.csv")

# Merge route metadata into historical train dataset
train_ridership = train_ridership.merge(bus_routes[['Route_ID', 'Route_Type', 'Route_Code']], on='Route_ID', how='left')

# Helper function: Group by Date & Route Code to get Total Daily Pax
def get_daily_route_pax(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Total_Pax'] = df['Boarding_Count'] + df['Alighting_Count']
    return df.groupby(['Date', 'Route_Code', 'Route_Type'])['Total_Pax'].sum().reset_index()

train_agg = get_daily_route_pax(train_ridership)
shock_agg = get_daily_route_pax(shock_ridership)
oot_agg   = get_daily_route_pax(oot_ridership)

combined_actuals = pd.concat([train_agg, shock_agg, oot_agg]).sort_values(by=['Route_Code', 'Date'])

# ----------------------------------------------------
# 2. Computations: Stage 3 Accountability Audit
# Evaluating Forecast (Stage 2 expectations) vs Actuals (Stage 3 OOT)
# ----------------------------------------------------
print("Computing Stage 3 Metrics...")

# H1 2025 Baseline
h1_2025 = train_agg[train_agg['Date'].dt.year == 2025].copy()
route_baseline_h1 = h1_2025.groupby('Route_Code')['Total_Pax'].mean().reset_index(name='Stage1_Expected_Q3')

# Q3 Post-Shock Recalibrated Forecast
# (Assuming recalibration stated the Q3 shock averages become the new expectation for Q4)
route_baseline_q3 = shock_agg.groupby('Route_Code')['Total_Pax'].mean().reset_index(name='Stage2_Expected_Q4')

# Actual Out of Time Data (Q4 Reality)
actual_q4 = oot_agg.groupby('Route_Code')['Total_Pax'].mean().reset_index(name='Actual_Q4')

# Merge for Audit DataFrame
audit = route_baseline_h1.merge(route_baseline_q3, on='Route_Code')
audit = audit.merge(actual_q4, on='Route_Code')
audit = audit.merge(bus_routes[['Route_Code', 'Route_Type']].drop_duplicates(), on='Route_Code')

# Stage 3 Metric Requirements
audit['MAPE_Q4'] = abs((audit['Actual_Q4'] - audit['Stage2_Expected_Q4']) / audit['Stage2_Expected_Q4']) * 100
audit['Directional_Bias_Q4'] = ((audit['Actual_Q4'] - audit['Stage2_Expected_Q4']) / audit['Stage2_Expected_Q4']) * 100

# Identify Structural Misclassification (Decay vs Sustained vs Underreaction)
conditions = [
    (audit['Directional_Bias_Q4'] > 5),
    (audit['Directional_Bias_Q4'] < -5)
]
choices = ['Underreaction (Growth Surpassed Shock)', 'Overreaction (Shock Decayed)']
audit['Accountability_Verdict'] = np.select(conditions, choices, default='Accurate (Stabilized/Sustained)')

print("\n--- Route-Level Aggregation KPIs ---")
print(audit.groupby('Route_Type')[['MAPE_Q4', 'Directional_Bias_Q4']].mean().round(2))

# ----------------------------------------------------
# 3. Premium Interactive Plotly Visualizations (fig.show())
# ----------------------------------------------------
print("Rendering Plotly Diagnostics...")

# --- CHART 1: Q4 Forecast Validation (Expected vs Actual) ---
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=audit['Route_Code'], y=audit['Stage2_Expected_Q4'],
    name='Recalibrated Forecast (from Q3 Models)', marker_color='rgb(55, 83, 109)'
))
fig1.add_trace(go.Bar(
    x=audit['Route_Code'], y=audit['Actual_Q4'],
    name='Actual Reality (Q4 Out-Of-Time)', marker_color='rgb(26, 118, 255)'
))
fig1.update_layout(
    title='Accountability Audit: Testing Stage 2 Forecasts Against Q4 Actuals',
    xaxis_title='Transport Corridors (Route Code)', yaxis_title='Average Daily Passengers',
    barmode='group', template='plotly_white', hovermode='x unified'
)
fig1.show()


# --- CHART 2: Structural Network Equilibrium Time-Series ---
# Charting just 2025 to see Baseline -> Shock -> Stabilization
agg_2025 = combined_actuals[combined_actuals['Date'].dt.year == 2025].copy()
agg_2025 = agg_2025.groupby(['Date', 'Route_Type'])['Total_Pax'].sum().reset_index()

# 7-day rolling average for smoother visualization of the equilibrium shift
agg_2025['Total_Pax_SMA'] = agg_2025.groupby('Route_Type')['Total_Pax'].transform(lambda x: x.rolling(7, min_periods=1).mean())

fig2 = px.line(
    agg_2025, x='Date', y='Total_Pax_SMA', color='Route_Type',
    title='Tracking the Regime Shift: H1 (Baseline) -> Q3 (Shock) -> Q4 (Stabilized Equilibrium)',
    labels={'Total_Pax_SMA': 'Smoothed Daily Transit Network Load'},
    line_shape='spline'
)

# Convert string dates to pandas Timestamps -> JS timestamps (required for plotly add_vline in some environments)
t_q3 = pd.Timestamp("2025-07-01").timestamp() * 1000
t_q4 = pd.Timestamp("2025-10-01").timestamp() * 1000

fig2.add_vline(x=t_q3, line_width=2, line_dash="dash", line_color="red", annotation_text="Shock Starts (Q3)")
fig2.add_vline(x=t_q4, line_width=2, line_dash="dash", line_color="green", annotation_text="OOT Validation Starts (Q4)")
fig2.update_layout(template='plotly_white', hovermode="x unified")
fig2.show()


# --- CHART 3: Quantifying Overreaction vs Underreaction Bias ---
fig3 = px.scatter(
    audit,
    x='Actual_Q4', y='Directional_Bias_Q4',
    color='Accountability_Verdict', text='Route_Code', size='MAPE_Q4',
    title='Diagnostic Matrix: Identifying Structural Misclassification (Bias vs Actuals)',
    labels={'Actual_Q4': 'Actual Q4 Daily Volume', 'Directional_Bias_Q4': 'Directional Error / Bias (%)', 'MAPE_Q4': 'Absolute Error'},
    color_discrete_map={
        'Accurate (Stabilized/Sustained)': 'green',
        'Overreaction (Shock Decayed)': 'orange',
        'Underreaction (Growth Surpassed Shock)': 'red'
    }
)
fig3.add_hline(y=0, line_dash="solid", line_color="black") # 0 Error baseline
fig3.update_traces(textposition='top center')
fig3.update_layout(template='plotly_white')
fig3.show()

Loading data...
Computing Stage 3 Metrics...

--- Route-Level Aggregation KPIs ---
            MAPE_Q4  Directional_Bias_Q4
Route_Type                              
City          14.05                14.05
Express        1.09                -1.09
Feeder         9.33                 9.33
Intercity     12.27                12.27
Rendering Plotly Diagnostics...
